# OmniVoice Robust Long-Form TTS — Google Colab

This notebook uses the current `master` branch. It provides semantic chunking, ASR verification, retry/splitting, and explicit-silence stitching for narration where exact wording matters.

Select **Runtime → Change runtime type → T4 GPU** before running.


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a T4 GPU first."

!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@master" soundfile


In [ ]:
from google.colab import files

uploaded = files.upload()
REF_AUDIO = next(iter(uploaded))
REF_TEXT = "PASTE THE EXACT REFERENCE TRANSCRIPT HERE."
assert "PASTE THE EXACT" not in REF_TEXT, "Set REF_TEXT before generating."
print("Reference:", REF_AUDIO)


In [ ]:
TEXT = """
Paste your long-form narration here.
""".strip()
assert TEXT and "Paste your" not in TEXT, "Set TEXT before generating."


In [ ]:
import torch
from omnivoice import (
    OmniVoice,
    OmniVoiceGenerationConfig,
    RobustLongFormConfig,
    RobustLongFormGenerator,
)

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
)

voice_prompt = model.create_voice_clone_prompt(
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,
    preprocess_prompt=True,
)

robust = RobustLongFormGenerator(
    model,
    RobustLongFormConfig(
        max_chunk_words=28,
        max_retries=3,
        max_split_depth=2,
        verify_with_asr=True,
        asr_model_name="openai/whisper-small.en",
        asr_device="cpu",
        strict=False,
    ),
)

generation_config = OmniVoiceGenerationConfig(
    num_step=32,
    guidance_scale=2.0,
    position_temperature=1.0,
    class_temperature=0.0,
)
print("Ready.")


In [ ]:
result = robust.generate(
    TEXT,
    language="en",
    voice_clone_prompt=voice_prompt,
    generation_config=generation_config,
)

print("All verified:", result.all_verified)
for i, report in enumerate(result.reports, 1):
    print(
        f"{i:02d} PASS={report.accepted} "
        f"WER={report.wer:.3f} attempts={report.attempts}"
    )
    print(" expected:", report.text)
    print(" ASR     :", report.transcript)


In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from google.colab import files

OUTPUT = "/content/omnivoice_robust_output.wav"
sf.write(OUTPUT, result.audio, result.sampling_rate)
display(Audio(OUTPUT))
files.download(OUTPUT)
